# OPERA DISP-S1 Data Explorer

This notebook explores all data types used in the displacement classification pipeline:

| Section | Data Type | What You'll See |
|---|---|---|
| 1 | OPERA DISP-S1 Granule | HDF5 structure, groups, datasets, coordinates, CRS |
| 2 | Displacement Layers | Spatial maps of displacement, coherence, quality masks |
| 3 | Copernicus GLO-30 DEM | Elevation, slope, aspect |
| 4 | Auxiliary Shapefiles | USGS faults and landslide inventory |
| 5 | Processed Features | Feature distributions, class balance, per-region stats |
| 6 | Time Series | Raw displacement curves by class |

**Run all cells top-to-bottom.** Update the CONFIG cell if your paths differ.

In [ ]:
# CONFIG — update if your paths differ

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path("..").resolve()

# Auto-find one sample OPERA file
SAMPLE_OPERA_FILE = next(
    (f for region_dir in (PROJECT_ROOT / "data").iterdir() if region_dir.is_dir()
     for f in region_dir.glob("*.nc")),
    None,
)

# Auto-find one DEM file
SAMPLE_DEM_FILE = next(
    (PROJECT_ROOT / "data" / "dem").glob("dem_*.tif"),
    None,
)

FAULT_SHAPEFILE = PROJECT_ROOT / "auxiliary" / "faults" / "Qfaults_US_Database.shp"
LANDSLIDE_SHAPEFILE = PROJECT_ROOT / "auxiliary" / "landslides" / "us_ls_v3_point.shp"
FEATURES_CSV = PROJECT_ROOT / "processed" / "features_all.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"OPERA sample: {SAMPLE_OPERA_FILE.name if SAMPLE_OPERA_FILE else 'NOT FOUND'}")
print(f"DEM sample:   {SAMPLE_DEM_FILE.name if SAMPLE_DEM_FILE else 'NOT FOUND'}")
print(f"Faults:       {'✓' if FAULT_SHAPEFILE.exists() else '✗'}")
print(f"Landslides:   {'✓' if LANDSLIDE_SHAPEFILE.exists() else '✗'}")
print(f"Features CSV: {'✓' if FEATURES_CSV.exists() else '✗'}")

---
## 1. OPERA DISP-S1 Granule Structure

Each OPERA DISP-S1 file is an HDF5 (netCDF4-compatible) file containing:

- **Root group `/`**: Displacement layers, quality masks, coordinate arrays
- **`/identification`**: Frame ID, reference/secondary dates, orbit direction, radar wavelength
- **`/corrections`**: Ionospheric delay, solid earth tides
- **`/metadata`**: Algorithm parameters, processing configuration

Key details:
- Coordinates (`/x`, `/y`) are **UTM meters**, not lat/lon
- The CRS lives in `spatial_ref.attrs['crs_wkt']` — the variable itself is a meaningless scalar
- Reference datetime is in `/identification`, not in root attributes
- `/y` values **decrease** (north to south), `/x` values increase (west to east)
- Pixel convention is "pixel is area" — coordinates refer to pixel centers

In [ ]:
import h5py

if SAMPLE_OPERA_FILE:
    print(f"File: {SAMPLE_OPERA_FILE.name}")
    print(f"Size: {SAMPLE_OPERA_FILE.stat().st_size / 1e6:.1f} MB")

    with h5py.File(SAMPLE_OPERA_FILE, "r") as f:

        # Root attributes
        print(f"\n--- Root Attributes ---")
        for k, v in f.attrs.items():
            val = v.decode() if isinstance(v, bytes) else v
            print(f"  {k}: {val}")

        # HDF5 tree
        print(f"\n--- HDF5 Tree ---")
        def print_tree(name, obj):
            indent = "  " + "  " * name.count("/")
            if isinstance(obj, h5py.Dataset):
                print(f"{indent}[dataset] {name}  shape={obj.shape}  dtype={obj.dtype}")
            elif isinstance(obj, h5py.Group):
                print(f"{indent}[group]   {name}/")
        f.visititems(print_tree)
else:
    print("No OPERA file found. Download data first.")

### `/identification` — The Important Metadata

This group contains the reference datetime, secondary datetime, frame ID, orbit direction,
and other critical metadata that the preprocessing pipeline reads via `h5py`.

In [ ]:
if SAMPLE_OPERA_FILE:
    with h5py.File(SAMPLE_OPERA_FILE, "r") as f:
        if "identification" in f:
            print("--- /identification ---")
            for key in f["identification"]:
                val = f["identification"][key][()]
                if isinstance(val, bytes):
                    val = val.decode()
                print(f"  {key}: {val}")

### Coordinate Arrays & CRS

- `/x` and `/y` are 1D arrays of UTM coordinates in meters
- `/y` decreases (row 0 is the northernmost point)
- The UTM zone varies by frame — always read from `spatial_ref.attrs`
- `spatial_ref` is a scalar int whose **attributes** hold the CRS, not the value itself

In [ ]:
if SAMPLE_OPERA_FILE:
    with h5py.File(SAMPLE_OPERA_FILE, "r") as f:
        x = f["x"][:]
        y = f["y"][:]
        print(f"/x: {len(x)} values, {x[0]:.1f} to {x[-1]:.1f} m  (spacing: {x[1]-x[0]:.1f} m)")
        print(f"/y: {len(y)} values, {y[0]:.1f} to {y[-1]:.1f} m  (spacing: {y[1]-y[0]:.1f} m)")
        print(f"Grid: {len(y)} rows x {len(x)} cols = {len(y)*len(x):,} pixels")

        print(f"\n--- spatial_ref ---")
        print(f"  Value: {f['spatial_ref'][()]} (ignore this)")
        for k, v in f["spatial_ref"].attrs.items():
            val = v.decode() if isinstance(v, bytes) else v
            if k == "crs_wkt":
                print(f"  {k}: {str(val)[:120]}...")
            else:
                print(f"  {k}: {val}")

---
## 2. Displacement & Quality Layers

The root group contains the data layers we use:

| Layer | Units | Description |
|---|---|---|
| `displacement` | meters | Cumulative LOS displacement (includes long-wavelength signals) |
| `short_wavelength_displacement` | meters | Local displacement (>30km signals removed) — **this is what we use** |
| `temporal_coherence` | 0–1 | Pixel reliability (higher = better) |
| `recommended_mask` | 0/1 | Binary quality mask (1 = good) |
| `connected_component_labels` | int | 0 = unreliable |

**Positive displacement = motion toward the satellite** (line-of-sight, not vertical).

In [ ]:
if SAMPLE_OPERA_FILE:
    import xarray as xr
    ds = xr.open_dataset(SAMPLE_OPERA_FILE, engine="h5netcdf")

    # Layer stats
    layers = ["displacement", "short_wavelength_displacement", "temporal_coherence",
              "recommended_mask", "connected_component_labels"]

    for layer in layers:
        if layer in ds:
            data = ds[layer].values.squeeze()
            valid = data[~np.isnan(data)] if np.issubdtype(data.dtype, np.floating) else data
            pct = 100 * len(valid) / data.size
            print(f"{layer}:")
            print(f"  shape={data.shape}  dtype={data.dtype}  valid={pct:.1f}%")
            if len(valid) > 0:
                print(f"  range: {np.min(valid):.6f} to {np.max(valid):.6f}  mean: {np.mean(valid):.6f}")
            print()

In [ ]:
if SAMPLE_OPERA_FILE:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f"OPERA DISP-S1: {SAMPLE_OPERA_FILE.name}", fontsize=11)

    disp = ds["short_wavelength_displacement"].values.squeeze()
    coh = ds["temporal_coherence"].values.squeeze()
    mask = ds["recommended_mask"].values.squeeze()

    # Short wavelength displacement
    im = axes[0,0].imshow(disp, cmap="RdBu", vmin=-0.05, vmax=0.05)
    axes[0,0].set_title("Short Wavelength Displacement (m)")
    plt.colorbar(im, ax=axes[0,0], fraction=0.046)

    # Full displacement
    if "displacement" in ds:
        full = ds["displacement"].values.squeeze()
        im = axes[0,1].imshow(full, cmap="RdBu", vmin=-0.05, vmax=0.05)
        axes[0,1].set_title("Full Displacement (m)")
        plt.colorbar(im, ax=axes[0,1], fraction=0.046)

    # Temporal coherence
    im = axes[0,2].imshow(coh, cmap="viridis", vmin=0, vmax=1)
    axes[0,2].set_title("Temporal Coherence")
    plt.colorbar(im, ax=axes[0,2], fraction=0.046)

    # Recommended mask
    axes[1,0].imshow(mask, cmap="gray")
    axes[1,0].set_title("Recommended Mask (1=good)")

    # Combined quality
    quality = (mask == 1) & (coh > 0.5)
    axes[1,1].imshow(quality, cmap="gray")
    pct = 100 * np.sum(quality) / quality.size
    axes[1,1].set_title(f"Quality: mask=1 & coh>0.5 ({pct:.1f}%)")

    # Histogram
    valid_disp = disp[quality & ~np.isnan(disp)]
    if len(valid_disp) > 0:
        axes[1,2].hist(valid_disp * 1000, bins=100, color="steelblue", edgecolor="none")
        axes[1,2].axvline(0, color="red", ls="--", alpha=0.5)
        axes[1,2].set_xlabel("Displacement (mm)")
        axes[1,2].set_title("Displacement Distribution (valid pixels)")

    plt.tight_layout()
    plt.show()
    ds.close()

---
## 3. Copernicus GLO-30 DEM

The DEM is a 30m resolution elevation grid from the Copernicus program, distributed as
1°×1° tiles on a public AWS bucket. `download_dem.py` stitches the tiles covering each
OPERA frame into a single GeoTIFF.

During preprocessing, `load_dem_for_frame` reprojects this from its native lat/lon grid
onto the exact OPERA UTM pixel grid so that pixel `[i,j]` in the DEM aligns with
pixel `[i,j]` in the displacement data.

From the elevation we compute:
- **Slope** (degrees): how steep the terrain is
- **Aspect** (sin/cos encoded): which compass direction the slope faces

In [ ]:
if SAMPLE_DEM_FILE:
    import rasterio

    print(f"File: {SAMPLE_DEM_FILE.name}")
    print(f"Size: {SAMPLE_DEM_FILE.stat().st_size / 1e6:.1f} MB")

    with rasterio.open(SAMPLE_DEM_FILE) as src:
        print(f"CRS: {src.crs}")
        print(f"Shape: {src.height} x {src.width}")
        print(f"Resolution: {src.res}")
        print(f"Bounds: {src.bounds}")
        elev = src.read(1)

    valid = elev[~np.isnan(elev)]
    print(f"Elevation: {np.min(valid):.0f} to {np.max(valid):.0f} m (mean {np.mean(valid):.0f} m)")
else:
    print("No DEM found. Run download_dem.py first.")

In [ ]:
if SAMPLE_DEM_FILE:
    # Compute slope
    dz_dy, dz_dx = np.gradient(elev, 30.0, 30.0)
    slope = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f"DEM: {SAMPLE_DEM_FILE.name}", fontsize=11)

    im = axes[0].imshow(elev, cmap="terrain",
                        vmin=np.nanpercentile(elev, 2),
                        vmax=np.nanpercentile(elev, 98))
    axes[0].set_title("Elevation (m)")
    plt.colorbar(im, ax=axes[0], fraction=0.046)

    im = axes[1].imshow(slope, cmap="YlOrRd", vmin=0, vmax=45)
    axes[1].set_title("Slope (degrees)")
    plt.colorbar(im, ax=axes[1], fraction=0.046)

    axes[2].hist(slope[~np.isnan(slope)].ravel(), bins=100, color="coral", edgecolor="none")
    axes[2].set_xlabel("Slope (degrees)")
    axes[2].set_title(f"Slope Distribution (mean={np.nanmean(slope):.1f}°)")
    axes[2].axvline(np.nanmean(slope), color="red", ls="--")

    plt.tight_layout()
    plt.show()

---
## 4. Auxiliary Shapefiles

Two shapefiles provide ground truth for spatial filtering:

- **USGS Quaternary Fault Database**: Line features representing known active faults.
  Used with a 2km buffer to select pixels near fault traces for the tectonic class.

- **USGS Landslide Inventory**: Point features marking known landslide locations.
  Used with a 500m buffer to select pixels in landslide-prone areas.

These provide **location-based labels** — independent ground truth that doesn't
depend on the displacement data itself (unlike rate-based filtering).

In [ ]:
import geopandas as gpd
from shapely.geometry import box

# --- Faults ---
if FAULT_SHAPEFILE.exists():
    faults = gpd.read_file(FAULT_SHAPEFILE)
    print(f"--- USGS Quaternary Fault Database ---")
    print(f"  Features: {len(faults)}")
    print(f"  CRS: {faults.crs}")
    print(f"  Geometry types: {faults.geom_type.value_counts().to_dict()}")
    print(f"  Columns: {list(faults.columns)}")

    if "fault_name" in faults.columns:
        print(f"\n  Pipeline faults:")
        for name in ["San Andreas", "Hayward", "Wasatch"]:
            matches = faults[faults["fault_name"].str.contains(name, case=False, na=False)]
            print(f"    '{name}': {len(matches)} features")
else:
    print(f"Fault shapefile not found: {FAULT_SHAPEFILE}")

In [ ]:
# --- Landslides ---
if LANDSLIDE_SHAPEFILE.exists():
    landslides = gpd.read_file(LANDSLIDE_SHAPEFILE)
    print(f"--- USGS Landslide Inventory ---")
    print(f"  Features: {len(landslides)}")
    print(f"  CRS: {landslides.crs}")
    print(f"  Columns: {list(landslides.columns)}")

    ls_4326 = landslides.to_crs("EPSG:4326")
    bboxes = {
        "oregon_coast":     (-124.2, 43.5, -123.5, 44.5),
        "california_north": (-124.5, 40.0, -123.5, 41.5),
        "washington":       (-123.0, 46.5, -122.0, 47.5),
    }
    print(f"\n  Landslide points in pipeline regions:")
    for name, bbox in bboxes.items():
        count = ls_4326[ls_4326.geometry.within(box(*bbox))].shape[0]
        print(f"    {name}: {count} points")
else:
    print(f"Landslide shapefile not found: {LANDSLIDE_SHAPEFILE}")

In [ ]:
# Plot both shapefiles (Western US)
if FAULT_SHAPEFILE.exists() and LANDSLIDE_SHAPEFILE.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    faults_4326 = faults.to_crs("EPSG:4326")
    west = faults_4326.cx[-126:-110, 30:50]
    west.plot(ax=axes[0], color="red", linewidth=0.3)
    axes[0].set_title(f"Quaternary Faults (Western US): {len(west)} features")
    axes[0].set_xlabel("Longitude")
    axes[0].set_ylabel("Latitude")

    west_ls = ls_4326.cx[-126:-110, 30:50]
    west_ls.plot(ax=axes[1], color="orange", markersize=0.5, alpha=0.3)
    axes[1].set_title(f"Landslide Inventory (Western US): {len(west_ls)} points")
    axes[1].set_xlabel("Longitude")
    axes[1].set_ylabel("Latitude")

    plt.tight_layout()
    plt.show()

---
## 5. Processed Features

After preprocessing, each pixel becomes a row in a CSV with 19 features:

**Temporal** (from the stitched displacement time series):
`linear_rate_mm_yr`, `r_squared_linear`, `total_disp_mm`, `max_abs_disp_mm`,
`acceleration`, `r2_ratio_quad_lin`, `seasonal_amp_mm`, `residual_std_mm`,
`autocorr_lag1`, `vel_sign_changes`, `vel_kurtosis`, `vel_skewness`

**Spatial** (from the neighborhood rate map):
`mean_coherence`, `nbr_mean_rate_mm_yr`, `nbr_std_rate_mm_yr`, `spatial_gradient`

**Terrain** (from the GLO-30 DEM):
`slope_deg`, `aspect_sin`, `aspect_cos`

In [ ]:
import pandas as pd

if FEATURES_CSV.exists():
    df = pd.read_csv(FEATURES_CSV)
    meta_cols = ["pixel_y", "pixel_x", "frame_id", "region", "label"]
    feat_cols = [c for c in df.columns if c not in meta_cols]

    print(f"Samples: {len(df)}")
    print(f"Features: {len(feat_cols)}")
    print(f"Columns: {list(df.columns)}")

    print(f"\n--- Class Balance ---")
    for label, count in df["label"].value_counts().items():
        regions = sorted(df[df["label"] == label]["region"].unique())
        print(f"  {label:<12s}  {count:>5d} samples  ({', '.join(regions)})")
else:
    print("No features file found. Run preprocessing first.")

In [ ]:
if FEATURES_CSV.exists():
    print(f"{'Feature':<28s} {'Min':>10s} {'Mean':>10s} {'Max':>10s} {'Std':>10s}")
    print("-" * 68)
    for col in feat_cols:
        v = df[col]
        print(f"{col:<28s} {v.min():>10.3f} {v.mean():>10.3f} {v.max():>10.3f} {v.std():>10.3f}")

### Feature Distributions by Class

Key features should show separation between classes. Look for:
- **Subsidence**: strongly negative `linear_rate_mm_yr`, high `autocorr_lag1`, low `slope_deg`
- **Tectonic**: both positive and negative rates (strike-slip), moderate `slope_deg`
- **Landslide**: steep `slope_deg`, lower coherence
- **Stable**: rates near zero, high coherence

In [ ]:
if FEATURES_CSV.exists():
    key_features = ["linear_rate_mm_yr", "slope_deg", "autocorr_lag1",
                    "seasonal_amp_mm", "mean_coherence", "nbr_std_rate_mm_yr"]
    key_features = [f for f in key_features if f in df.columns]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle("Feature Distributions by Class", fontsize=13)
    axes = axes.ravel()

    for i, feat in enumerate(key_features):
        for label in sorted(df["label"].unique()):
            vals = df[df["label"] == label][feat].dropna()
            axes[i].hist(vals, bins=50, alpha=0.5, label=label, density=True)
        axes[i].set_xlabel(feat)
        axes[i].set_ylabel("Density")
        axes[i].legend(fontsize=7)

    for j in range(len(key_features), len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()

### Per-Region Feature Comparison

Regions within the same class should have similar feature distributions.
If they don't, the model may struggle to generalize across regions.

In [ ]:
if FEATURES_CSV.exists():
    print(f"--- linear_rate_mm_yr by Region ---")
    for region in sorted(df["region"].unique()):
        vals = df[df["region"] == region]["linear_rate_mm_yr"]
        label = df[df["region"] == region]["label"].iloc[0]
        print(f"  {region:<20s} ({label:<12s}): {vals.mean():>8.2f} ± {vals.std():>7.2f} mm/yr")

    if "slope_deg" in df.columns:
        print(f"\n--- slope_deg by Region ---")
        for region in sorted(df["region"].unique()):
            vals = df[df["region"] == region]["slope_deg"]
            label = df[df["region"] == region]["label"].iloc[0]
            print(f"  {region:<20s} ({label:<12s}): {vals.mean():>8.2f} ± {vals.std():>7.2f}°")

---
## 6. Sample Time Series

The raw displacement time series show what the model is learning from.
Each curve is one pixel's displacement over time (in mm).

Look for:
- **Subsidence**: steady downward trend
- **Tectonic**: steady linear trend (positive or negative depending on LOS geometry)
- **Landslide**: noisier, possibly seasonal, on steep terrain
- **Stable**: flat, near zero, low noise

In [ ]:
ts_dir = PROJECT_ROOT / "processed"
npz_files = sorted(ts_dir.glob("timeseries_*.npz"))

if npz_files:
    regions_with_ts = []
    for npz_path in npz_files:
        region = npz_path.stem.replace("timeseries_", "")
        data = np.load(npz_path)
        regions_with_ts.append((region, data["time_series"], data["t_days"]))

    n = len(regions_with_ts)
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows))
    if n == 1:
        axes = np.array([axes])
    axes = axes.ravel()

    fig.suptitle("Sample Time Series (5 random pixels per region)", fontsize=13)

    for i, (region, ts, t_days) in enumerate(regions_with_ts):
        ax = axes[i]
        if FEATURES_CSV.exists() and region in df["region"].values:
            label = df[df["region"] == region]["label"].iloc[0]
        else:
            label = "?"

        n_show = min(5, len(ts))
        idx = np.random.choice(len(ts), n_show, replace=False)
        for j in idx:
            ax.plot(t_days / 365.25, ts[j] * 1000, alpha=0.6, linewidth=0.8)

        ax.set_xlabel("Years")
        ax.set_ylabel("Displacement (mm)")
        ax.set_title(f"{region} ({label})\n{len(ts)} pixels, {len(t_days)} timesteps")
        ax.axhline(0, color="gray", ls="--", lw=0.5)

    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()
else:
    print("No time series files found. Run preprocessing first.")

---
## Summary

Quick inventory of what's available on disk.

In [ ]:
data_dir = PROJECT_ROOT / "data"
proc_dir = PROJECT_ROOT / "processed"

print("=" * 50)
print("  DATA INVENTORY")
print("=" * 50)

if data_dir.exists():
    region_dirs = [d for d in data_dir.iterdir() if d.is_dir() and d.name != "dem"]
    nc_count = sum(len(list(d.glob("*.nc"))) for d in region_dirs)
    dem_count = len(list((data_dir / "dem").glob("*.tif"))) if (data_dir / "dem").exists() else 0

    print(f"\n  Raw data:")
    print(f"    {len(region_dirs)} region directories")
    print(f"    {nc_count} OPERA granules (.nc)")
    print(f"    {dem_count} DEM files (.tif)")

    for d in sorted(region_dirs):
        n = len(list(d.glob("*.nc")))
        if n > 0:
            print(f"      {d.name}: {n} files")

if proc_dir.exists():
    csv_count = len(list(proc_dir.glob("features_*.csv"))) - (1 if (proc_dir / "features_all.csv").exists() else 0)
    npz_count = len(list(proc_dir.glob("timeseries_*.npz")))
    print(f"\n  Processed:")
    print(f"    {csv_count} regional feature CSVs")
    print(f"    {npz_count} time series archives")
    if FEATURES_CSV.exists():
        df_summary = pd.read_csv(FEATURES_CSV)
        print(f"    {len(df_summary)} total samples in features_all.csv")
        print(f"    {df_summary['label'].nunique()} classes, {df_summary['region'].nunique()} regions")

print(f"\n  Auxiliary:")
print(f"    Faults:     {'✓' if FAULT_SHAPEFILE.exists() else '✗'}")
print(f"    Landslides: {'✓' if LANDSLIDE_SHAPEFILE.exists() else '✗'}")